# 02 - Production EDA (SYNTHETIC)
Questions: where is production lost, and which wells decline or water out fastest?

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30); pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
from og_oip import config
NOTICE = config.SYNTHETIC_NOTICE
print(NOTICE)

All operational, production, financial, maintenance, inventory, sensor and HSE data in this project are synthetic/simulated and created for portfolio demonstration purposes. They do not represent actual operations of a real company.


In [2]:
M = {p.stem: pd.read_csv(p, parse_dates=[c for c in ("date","month") if c in pd.read_csv(p, nrows=0).columns]) for p in config.MARTS_DIR.glob("mart_*.csv")}
wd, fm, es = M["mart_well_daily"], M["mart_field_monthly"], M["mart_equipment_summary"]

In [3]:
from og_oip.analytics import production
k = production.portfolio_kpis(wd)
print(f"Oil {k['total_oil_bbl']/1e6:.2f} MMbbl | loss {k['production_loss_pct']:.1f}% of simulated potential | downtime share of loss {k['downtime_share_of_loss_pct']:.0f}%")
print(f"Downtime hours associated with maintenance events: {k['maintenance_associated_downtime_share_pct']:.0f}%")

Oil 13.35 MMbbl | loss 9.4% of simulated potential | downtime share of loss 67%
Downtime hours associated with maintenance events: 93%


In [4]:
from og_oip.reporting import charts
fig, ax = plt.subplots(figsize=(9,4)); p = fm.pivot_table(index="month", columns="field_id", values="oil_bbl", aggfunc="sum")/1e3
ax.stackplot(p.index, p.T.values, labels=p.columns); ax.legend(); ax.set_title("Monthly oil by field (k bbl) - synthetic"); plt.show()

In [5]:
fields = pd.read_csv(config.PROCESSED_DIR / "dim_field.csv")
production.by_field(wd, fields)[["field_id","field_name","oil_bbl","loss_bbl","loss_pct","water_cut_pct"]]

,field_id,field_name,oil_bbl,loss_bbl,loss_pct,water_cut_pct
1,FIELD-002,Synthetic Field Bravo,"3,937,987.83","447,887.34",10.21,38.57
0,FIELD-001,Synthetic Field Alpha,"4,358,664.55","417,362.02",8.74,48.58
2,FIELD-003,Synthetic Field Charlie,"2,708,874.33","282,290.10",9.44,36.81
3,FIELD-004,Synthetic Field Delta,"2,346,772.35","236,589.44",9.16,51.49


### Well ranking
Ranking metrics are explicit: `rank_by_loss_bbl` (total barrels lost) and `rank_by_loss_pct` (loss / potential).

In [6]:
production.by_well(wd).head(10)[["well_name","field_id","loss_bbl","loss_pct","avg_oil_rate_bbl_d","rank_by_loss_bbl","rank_by_loss_pct"]]

,well_name,field_id,loss_bbl,loss_pct,avg_oil_rate_bbl_d,rank_by_loss_bbl,rank_by_loss_pct
1,PNX-102,FIELD-001,"161,629.30",8.33,"1,623.64",1,58
26,PNX-209,FIELD-002,"68,158.08",9.48,876.74,2,24
30,PNX-213,FIELD-002,"61,517.94",10.87,460.26,3,2
20,PNX-203,FIELD-002,"58,768.49",9.66,687.69,4,18
55,PNX-408,FIELD-004,"49,450.64",9.58,"1,021.10",5,20
36,PNX-303,FIELD-003,"49,130.27",9.93,925.01,6,16
21,PNX-204,FIELD-002,"45,409.06",10.15,390.97,7,12
8,PNX-109,FIELD-001,"45,314.15",8.48,446.49,8,53
32,PNX-215,FIELD-002,"42,972.32",10.11,394.35,9,13
31,PNX-214,FIELD-002,"40,424.30",11.17,566.11,10,1


In [7]:
dec = production.decline_analysis(wd)
print("wells with a decline fit:", len(dec), "| median nominal annual decline %:", round(dec.nominal_annual_decline_pct.median(),1))
dec.head(8)

wells with a decline fit: 55 | median nominal annual decline %: 15.1


,well_id,well_name,field_id,months_used,start_rate_bbl_d,end_rate_bbl_d,nominal_annual_decline_pct,r_squared,slope_p_value
3,WELL-0004,PNX-104,FIELD-001,30,479.62,148.46,39.93,0.99,0.00
19,WELL-0021,PNX-203,FIELD-002,26,"1,016.65",461.40,33.88,0.94,0.00
20,WELL-0022,PNX-204,FIELD-002,34,670.76,230.77,33.74,0.97,0.00
44,WELL-0048,PNX-314,FIELD-003,23,658.80,338.28,33.07,0.96,0.00
10,WELL-0011,PNX-111,FIELD-001,21,372.06,214.34,30.45,0.92,0.00
30,WELL-0032,PNX-214,FIELD-002,18,703.04,453.33,30.17,0.88,0.00
21,WELL-0023,PNX-205,FIELD-002,34,524.03,188.15,30.13,0.92,0.00
33,WELL-0035,PNX-301,FIELD-003,27,431.80,215.74,29.87,0.97,0.00


Decline fits are indicative: workovers and downtime create step changes that a single exponential does not capture.

In [8]:
la = production.loss_associations(wd); la

,target,factor,spearman_rho,p_value,n
0,loss_pct,avg water cut %,-0.05,0.04,1935
1,loss_pct,avg pressure psi,0.03,0.20,1935
2,loss_pct,well age (days),-0.03,0.17,1935
3,loss_pct,downtime hours in month,0.98,0.00,1935
4,other_loss_pct,avg water cut %,-0.00,0.93,1935
5,other_loss_pct,avg pressure psi,0.00,0.89,1935
6,other_loss_pct,well age (days),-0.00,0.99,1935
7,other_loss_pct,downtime hours in month,-0.22,0.00,1935


Spearman correlations are **associations only**. The strong correlation with downtime hours is by construction (loss is computed from downtime).